# Prep for earthmover

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import re
import glob
import zarr
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed
import arraylake

from funcs_support import get_filepaths, get_params
from funcs_aux import _load_gwls
dir_list = get_params()

In [2]:
# Log into arraylake
client = arraylake.Client()
client.login()

🔓 Successfully refreshed tokens!

> Token stored at /glade/u/home/schwarzwald/.arraylake/token.json

╭──────────────────────────────────────────────── 👤 User Details ────────────────────────────────────────────────╮
│ Name: None None                                                                                                 │
│ Email: ks3753@columbia.edu                                                                                      │
│ Id: eed8e278-d956-43f1-aa47-8472019241ef                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🔓 Successfully logged in!

> Token stored at /glade/u/home/schwarzwald/.arraylake/token.json

╭──────────────────────────────────────────────── 👤 User Details ────────────────────────────────────────────────╮
│ Name: None None                                                                                                 │
│ Email: ks3753@columbia.edu                                                                                      │
│ Id: eed8e278-d956-43f1-aa47-8472019241ef                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
#cluster = distributed.Client(
#    n_workers=6,
#    threads_per_worker=3,
#    processes=True,
#    memory_limit="8GB",  # or lower
#    dashboard_address=None,
#)
#display(cluster)

## From the finished `/bcd_me/` files

In [2]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['final']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
modlist = bcd_fns['model'].unique()
bcd_fns = bcd_fns.set_index(['typ','model'])

In [7]:
def process_ds(ds):
    ''' Place to put any changes made to the file before uploading '''
    ds = ds.sel(proj_base = [mod for mod in ds.proj_base.values if mod != 'PRISM'])
    ds.attrs['DESCRIPTION'] = re.sub(', PRISM','',ds.attrs['DESCRIPTION'])

    ds.attrs['Version'] = 'v1.0'
    return ds
    

In [11]:
org = 'ClimateUncertaintyLab'

overwrite = True

#for keys in bcd_fns.index:
for keys in [('qdm-qplad','EC-Earth3')]:
    
    typ = keys[0]
    mod = keys[1]
    
    # Set repo name (and create, if necessary)
    repo_name = f'bcd_me_{re.sub(r'\-','_',typ)}'
    if repo_name not in [k.name for k in client.list_repos(org)]:
        client.create_repo(org+'/'+repo_name)

    # Get session in BCD-ME repo
    repo = client.get_repo(org+'/'+repo_name)
    session = repo.writable_session('main')
    
    # Set group name (i.e., "file"name, in zarr speak)
    group = mod

    # Query group structure
    if not overwrite: 
        try:
            root = zarr.open_group(session.store, mode="r")
            if group in list(root.group_keys()):
                print(f'Group {mod} already created, skipped.')
                continue
        except:
            # (raises GroupNotFoundError if no groups)
            pass
    
    # Write commit message
    if typ == 'qdm':
        message = f'write v1.0 bias-corrected (QDM) {mod} data, with tas and tasmax'
    elif typ == 'qdm-qplad':
        message = f'write v1.0 downscaled (QDM-QPLAD) {mod} data, with tas- and tasmax-based variables'
    else:
        raise KeyError(f'typ is {str(typ)} instead of "qdm" or "qdm-qplad"')
    
    # Load zarr
    ds = xr.open_zarr(bcd_fns.loc[keys]['path'],decode_timedelta=False)

    ds = process_ds(ds)
    
    #---- Write!
    with repo.transaction('main', message=message) as store:
        with ProgressBar():
            # Save all runs / bias-corrected files for a model 
            ds.drop_encoding().to_zarr(store, group=group,compute=True,mode='w')
    print(f'{str(keys)} processed!')

/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in

[########################################] | 100% Completed | 29m 14s
('qdm-qplad', 'EC-Earth3') processed!


In [5]:
ds = xr.open_zarr(bcd_fns.loc[keys]['path'],decode_timedelta=False)

In [6]:
ds

<xarray.Dataset> Size: 1TB
Dimensions:        (idv: 35, proj_base: 4, lat: 568, lon: 1440, gwl: 8,
                    binC: 73, binF: 27, degree: 4, bnds: 2, year: 20,
                    variable: 6)
Coordinates: (12/17)
    experiment     (idv) <U6 840B dask.array<chunksize=(35,), meta=np.ndarray>
    model          (idv) <U9 1kB dask.array<chunksize=(35,), meta=np.ndarray>
    run            (idv) <U9 1kB dask.array<chunksize=(35,), meta=np.ndarray>
  * proj_base      (proj_base) <U6 96B 'ERA5' 'GMFD' 'JRA-3Q' 'MERRA2'
  * lat            (lat) float64 5kB -55.88 -55.62 -55.38 ... 85.38 85.62 85.88
  * lon            (lon) float64 12kB -179.9 -179.6 -179.4 ... 179.4 179.6 179.9
    ...             ...
  * bnds           (bnds) int64 16B 0 1
    binC_bnds      (binC, bnds) float64 1kB dask.array<chunksize=(73, 2), meta=np.ndarray>
    binF_bnds      (binF, bnds) float64 432B dask.array<chunksize=(27, 2), meta=np.ndarray>
  * year           (year) int64 160B 1 2 3 4 5 6 7 8 ... 13 14 15 16 17 18 19 20
    calendar_year  (idv, gwl, year) float64 45kB dask.array<chunksize=(35, 8, 20), meta=np.ndarray>
  * variable       (variable) object 48B 'tasmaxsumpoly' ... 'tasbinF'
Dimensions without coordinates: idv
Data variables:
    tasbinC        (idv, proj_base, lat, lon, gwl, binC) float64 535GB dask.array<chunksize=(1, 1, 200, 250, 1, 73), meta=np.ndarray>
    tasbinF        (idv, proj_base, lat, lon, gwl, binF) float64 198GB dask.array<chunksize=(1, 1, 200, 250, 1, 27), meta=np.ndarray>
    tasmaxbinC     (idv, proj_base, lat, lon, gwl, binC) float64 535GB dask.array<chunksize=(1, 1, 200, 250, 1, 73), meta=np.ndarray>
    tasmaxbinF     (idv, proj_base, lat, lon, gwl, binF) float64 198GB dask.array<chunksize=(1, 1, 200, 250, 1, 27), meta=np.ndarray>
    tasmaxsumpoly  (idv, proj_base, lat, lon, gwl, degree) float32 15GB dask.array<chunksize=(1, 1, 200, 250, 1, 4), meta=np.ndarray>
    tassumpoly     (idv, proj_base, lat, lon, gwl, degree) float32 15GB dask.array<chunksize=(1, 1, 200, 250, 1, 4), meta=np.ndarray>
Attributes:
    DESCRIPTION:  EC-Earth3 statistics of ScenarioMIP runs, bias-corrected us...
    Project:      Bias-Corrected, Downscaled Massive Ensemble (BCD-ME)
    Creators:     Kevin Schwarzwald, Nathan Lenssen, Radley Horton, and Gerno...
    Version:      v0.2
    License:      CC BY 4.0

## From the old separate local file structure in `dir_list['proc']`

In [6]:
org = 'ClimateUncertaintyLab'
repo_name = 'bcd_me_qm_qplad'

if repo_name not in [k.name for k in client.list_repos(org)]:
    client.create_repo(org+'/'+repo_name)

In [7]:
# Old file structure is QM-QPLAD, despite the name. This is separate from what's in 
# /bcd_me/, which is what the dir_list now is
df = get_filepaths(source_dir = 'proc',dir_list={'proc':'/glade/derecho/scratch/schwarzwald/climate_proc/'})
df = df.query("proj_method == 'QDM' and dwnscl_method == 'QPLAD' and proj_base != 'ERA5'") # Taking out ISIMIP ERA5

var = 'tas'
varlist = [var+'binC',var+'binF',var+'sumpoly']
df = df.iloc[[v in varlist for v in df.varname.values],:]

In [ ]:


modlist = np.unique(df.model.values)
for mod in modlist:
#for mod in [modlist[0]]:
#for mod in ['CESM2-WACCM']: # to test; smallest ensemble

    print(f'Processing {mod}!')

    # Get session in BCD-ME repo
    repo = client.get_repo(org+'/'+repo_name)
    session = repo.writable_session('main')
    
    # set group name (i.e., "file"name, in zarr speak)
    group = mod
    
    # Query group structure
    try:
        root = zarr.open_group(session.store, mode="r")
        if group in list(root.group_keys()):
            print(f'Group {mod} already created, skipped.')
            continue
    except:
        # (raises GroupNotFoundError if no groups)
        pass
    
    # Get filenames for a particular model
    df_tmp = df.query(f'model == "{mod}" and filetype == "zarr"') # for some reason, a few nc files got snuck in?

    # Load into single dask store
    dss = {var:xr.concat([load_for_concat(row,chunks={'lat':100,'lon':500,'gwl':1}) for row in df_tmp.query(f'varname == "{var}"').iterrows()],
          dim='idv',
          data_vars='all',
          join='outer')
          for var in np.unique(df_tmp.varname.values)}

    dss[var+'binC'] = dss[var+'binC'].rename({'bin':'binC','bin_bnds':'binC_bnds'})
    dss[var+'binF'] = dss[var+'binF'].rename({'bin':'binF','bin_bnds':'binF_bnds'})
    
    dss = xr.merge([d.sortby(d.idv) for v,d in dss.items()],join='outer')
    
    dss = dss.set_coords(['binC_bnds','binF_bnds'])


    # Reset index for export (no multiindices supported)
    dss = dss.reset_index('idv')
    
    # Rename the "ERA5-025" (originally in contrast to ISIMIP ERA5, 
    # which was falsely named as just "ERA5")
    dss['proj_base'] = [re.sub(r'ERA5\-025','ERA5',m) for m in dss.proj_base.values]
    
    #---- Metadata
     # Set variable-level metadata
    if var == 'tas':
        vardesc = 'Mean Near-Surface Air Temperature'
    elif var == 'tasmax':
        vardesc = 'Maximum Near-Surface Air Temperature'
        
    dss[var+'binC'].attrs =  {'long_name':f'Days With {vardesc} in Celsius Bins',
                                 'standard_name':'C_bin_days',
                                 'units':'days'}
    dss[var+'binF'].attrs =  {'long_name':f'Days With {vardesc} in Fahrenheit Bins',
                                 'standard_name':'F_bin_days',
                                 'units':'days'}
    dss[var+'sumpoly'].attrs =  {'long_name':f'Polynomial Sums of {vardesc} Per Year',
                                 'standard_name':var+'_sum_poly',
                                 'units':'C^k'}
  
    
    # Set coordinate-level metadata
    dss.binF.attrs = {'long_name':'Fahrenheit Bins','short_name':'bins'}
    dss.binF_bnds.attrs = {'long_name':'Fahrenheit Bin Bounds','short_name':'bin_bnds'}
    dss.binC.attrs = {'long_name':'Celsius Bins','short_name':'bins'}
    dss.binC_bnds.attrs = {'long_name':'Celsius Bin Bounds','short_name':'bin_bnds'}
    dss.degree.attrs = {'long_name':'polynomial degree','short_name':'poly_deg'}
    
    dss.experiment.attrs = {'long_name':'Experiment',
                            'description':'ScenarioMIP Experiment'}
    dss.run.attrs = {'long_name':'Ensemble Member',
                     'description':'Ensemble member ID'}
    dss.proj_base.attrs = {'long_name':'Bias-correction base reanalysis',
                           'description':'Reanalysis product used as "ground truth" for bias-correction'}
    dss.gwl.attrs = {'long_name':'Global Warming Level',
                     'short_name':'GWL',
                     'description':"Global mean surface temperature of this 20-year segment as anomalies above model run's 1850-1900 average"}
    
    # Set file metadata
    dss.attrs = {
        'DESCRIPTION':f'Statistics of {mod} ScenarioMIP runs, bias-corrected using Quantile Mapping (QM, Cannon et al. 2015) and downscaled using Quantile-Preserving Localized Analog-Downscaling (QPLAD, Gergel et al. 2024) to {', '.join(np.unique([dss.proj_base.values]))}',
        'Project':'Bias-Corrected, Downscaled Massive Ensemble (BCD-ME)',
        'Creators':'Kevin Schwarzwald, Nathan Lenssen, Radley Horton, and Gernot Wagner',
        'Version':'v0.1',
        'License':'CC BY-NC 4.0'
            }

    #---- Write!
    with repo.transaction('main', message='write downscaled (QM-QPLAD) '+mod+' data') as store:
        with ProgressBar():
            # Save all runs / bias-corrected files for a model 
            dss.drop_encoding().to_zarr(store, group=group,compute=True,mode='w')

Processing ACCESS-ESM1-5!
Group ACCESS-ESM1-5 already created, skipped.
Processing CESM2-WACCM!
Group CESM2-WACCM already created, skipped.
Processing CNRM-CM6-1!
Group CNRM-CM6-1 already created, skipped.
Processing CNRM-ESM2-1!
Group CNRM-ESM2-1 already created, skipped.
Processing CanESM5!
Group CanESM5 already created, skipped.
Processing EC-Earth3!


/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=9, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/glade/u/home/schwarzwald/.conda/envs/bcd_me/lib/python3.13/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=6, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in

[                                        ] | 0% Completed | 513.64 ms